In [2]:
import os
import pandas as pd
import snowflake.connector
from dotenv import load_dotenv

load_dotenv()

conn = snowflake.connector.connect(
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    role=os.getenv("SNOWFLAKE_ROLE"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
    database="RAW",
)

def run_query(sql):
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [desc[0] for desc in cur.description]
    cur.close()
    return pd.DataFrame(rows, columns=cols)

print("Connected!")

Connected!


In [3]:
# What does the current SOURCE breakdown look like?
run_query("""
    SELECT SOURCE, COUNT(*) as cnt
    FROM THEIRSTACK.SRC_POSTINGS
    GROUP BY SOURCE
    ORDER BY SOURCE
""")

,SOURCE,CNT
0,theirstack,33


In [4]:
# Preview what each regex would match before committing
run_query("""
    SELECT 
        RAW_PAYLOAD:job_title::STRING as job_title,
        CASE
            WHEN REGEXP_LIKE(LOWER(RAW_PAYLOAD:job_title::STRING), '.*data analyst.*') 
                THEN 'theirstack:Data Analyst'
            WHEN REGEXP_LIKE(LOWER(RAW_PAYLOAD:job_title::STRING), '.*analytics engineer.*') 
                THEN 'theirstack:Analytics Engineer'
            WHEN REGEXP_LIKE(LOWER(RAW_PAYLOAD:job_title::STRING), '.*data engineer.*') 
                THEN 'theirstack:Data Engineer'
            ELSE 'theirstack:Unknown'
        END as inferred_source
    FROM THEIRSTACK.SRC_POSTINGS
    WHERE SOURCE = 'theirstack'
    ORDER BY inferred_source
""")

,JOB_TITLE,INFERRED_SOURCE
0,"Analytics Engineer, Service Ops Analytics & AI",theirstack:Analytics Engineer
1,Analytics Engineer,theirstack:Analytics Engineer
2,Data Analyst,theirstack:Data Analyst
3,Data Analyst,theirstack:Data Analyst
4,Data Analyst - Consumer Datasets,theirstack:Data Analyst
5,Data Analyst (Entry-Level / Junior),theirstack:Data Analyst
6,Legal Assistant / Data Analyst Supporting the ...,theirstack:Data Analyst
7,Data Analyst,theirstack:Data Analyst
8,Business Data Analyst,theirstack:Data Analyst
9,"Data Analyst (FGP) - Manhattan, Central Billin...",theirstack:Data Analyst


In [5]:
# Summary of what the backfill would produce
run_query("""
    SELECT 
        CASE
            WHEN REGEXP_LIKE(LOWER(RAW_PAYLOAD:job_title::STRING), '.*data analyst.*') 
                THEN 'theirstack:Data Analyst'
            WHEN REGEXP_LIKE(LOWER(RAW_PAYLOAD:job_title::STRING), '.*analytics engineer.*') 
                THEN 'theirstack:Analytics Engineer'
            WHEN REGEXP_LIKE(LOWER(RAW_PAYLOAD:job_title::STRING), '.*data engineer.*') 
                THEN 'theirstack:Data Engineer'
            ELSE 'theirstack:Unknown'
        END as inferred_source,
        COUNT(*) as cnt
    FROM THEIRSTACK.SRC_POSTINGS
    WHERE SOURCE = 'theirstack'
    GROUP BY 1
    ORDER BY 1
""")

,INFERRED_SOURCE,CNT
0,theirstack:Analytics Engineer,2
1,theirstack:Data Analyst,31
